# Laboratório de metaheurísticas: Mochila 0-1

Este notebook explica as **peças** de `metaheuristics/utils/` e roda a linha de base (SA, ILS, VNS, ALNS, Busca Tabu, GRASP, PSO, AG e ACO puros). As misturas ficam por sua conta, na última seção.

Rode de cima para baixo, com o kernel do ambiente do Poetry (`poetry install` na raiz do repositório). O catálogo completo das peças está em [`utils/README.md`](utils/README.md).

In [ ]:
import random

from metaheuristics.adaptive_large_neighborhood_search import adaptive_large_neighborhood_search_mochila
from metaheuristics.ant_colony_optimization import ant_colony_optimization_mochila
from metaheuristics.genetic_algorithm import genetic_algorithm_mochila
from metaheuristics.grasp import grasp_mochila
from metaheuristics.iterated_local_search import iterated_local_search_mochila
from metaheuristics.particle_swarm_optimization import particle_swarm_optimization_mochila
from metaheuristics.simulated_annealing import simulated_annealing_mochila
from metaheuristics.tabu_search import tabu_search_mochila
from metaheuristics.utils import binaria
from metaheuristics.utils import ordem
from metaheuristics.utils.aceitacao import aceita_melhora
from metaheuristics.utils.aceitacao import metropolis
from metaheuristics.utils.aceitacao import temperaturas
from metaheuristics.utils.debug import Depurador
from metaheuristics.utils.experimento import resumir
from metaheuristics.utils.experimento import rodar_sementes
from metaheuristics.utils.experimento import tabela
from metaheuristics.utils.feromonio import Feromonio
from metaheuristics.utils.instancia import Item
from metaheuristics.utils.instancia import gerar_instancia_aleatoria
from metaheuristics.utils.instancia import programacao_dinamica
from metaheuristics.utils.plot import plotar_comparacao
from metaheuristics.utils.plot import plotar_historico
from metaheuristics.utils.selecao import selecao_roleta
from metaheuristics.utils.selecao import selecao_torneio
from metaheuristics.variable_neighborhood_search import variable_neighborhood_search_mochila

## 1. A instância

A instância são os dados de entrada: os itens (peso e valor) e a capacidade. Ela não muda enquanto o algoritmo roda.

- **Comum:** peso e valor sorteados separadamente. Há itens claramente bons e claramente ruins, e por isso quase qualquer método acerta.
- **Correlacionada:** o valor acompanha o peso (`valor = peso + 2`, a instância "fortemente correlacionada" de Pisinger). Nenhum item é óbvio, e o difícil passa a ser **encaixar pesos** para encher a mochila. É nela que as combinações se diferenciam.

O ótimo de cada uma vem da programação dinâmica e serve para calcular o gap.

In [ ]:
N_ITENS = 50

random.seed(42)
itens_comum = gerar_instancia_aleatoria(N_ITENS, 20, 100)
random.seed(42)
itens = gerar_instancia_aleatoria(N_ITENS, 20, 100, correlacionada=True)

capacidade_comum = sum(item.peso for item in itens_comum) // 2
capacidade = sum(item.peso for item in itens) // 2
otimo_comum, _ = programacao_dinamica(itens_comum, capacidade_comum)
otimo, _ = programacao_dinamica(itens, capacidade)

print("comum:          ", itens_comum[:5], "... C =", capacidade_comum, "· ótimo", otimo_comum)
print("correlacionada: ", itens[:5], "... C =", capacidade, "· ótimo", otimo)

## 2. As duas representações

A mesma solução (quais itens estão na mochila) pode ser escrita de dois jeitos:

| | Binária | Por ordem |
|---|---|---|
| Solução | `[1, 0, 0, 1, 0, 0, 0]`: um bit por item | `[0, 3, 1, 2, ...]`: a ordem em que se tenta colocar os itens |
| Análoga a | "por arestas" no caixeiro viajante | "por vértices" no caixeiro viajante |
| Viabilidade | pode passar da capacidade: precisa de reparo ou valor 0 | sempre viável: o decodificador só coloca o que cabe |
| Redundância | cada solução é um vetor só | ordens diferentes dão a mesma solução |

O exemplo abaixo usa a instância do README do guloso (C = 9). Os índices começam em 0, então o "item 1" do README é o índice 0.

In [ ]:
exemplo = [Item(p, v) for p, v in zip([4, 3, 7, 5, 3, 2, 3], [12, 8, 17, 11, 6, 2, 2])]

# a mesma solução nas duas representações: itens 1 e 4, valor 23
print(binaria.calcular_valor_peso([1, 0, 0, 1, 0, 0, 0], exemplo, 9))
print(ordem.decodificar([0, 3, 1, 2, 4, 5, 6], exemplo, 9), ordem.avaliar([0, 3, 1, 2, 4, 5, 6], exemplo, 9))

# redundância: outra ordem, a mesma solução
print(ordem.decodificar([3, 0, 6, 5, 4, 2, 1], exemplo, 9))

# na binária, um vetor pode passar da capacidade (valor 0); a perturbação repara
print(binaria.calcular_valor_peso([1, 1, 1, 0, 0, 0, 0], exemplo, 9))
random.seed(0)
print(binaria.perturbar([1, 0, 0, 1, 0, 0, 0], exemplo, 9, k=3))

## 3. As peças

Peças com o mesmo **encaixe** (o que recebem e o que devolvem) e a mesma representação são trocáveis. As duas representações usam os mesmos nomes: trocar de representação é trocar o prefixo `binaria.` por `ordem.`. Aceitação, seleção, feromônio, histórico, gráficos e debug servem para as duas representações.

| Encaixe | Binária | Por ordem |
|---|---|---|
| avaliar `(solucao, itens, C) → valor` | `binaria.avaliar`, `binaria.avaliar_penalizada` | `ordem.avaliar` |
| construtiva `(itens, C) → solucao` | `binaria.construtiva_aleatoria`, `binaria.construtiva_gulosa`, `binaria.construtiva_gulosa_aleatoria` (GRASP) | `ordem.construtiva_aleatoria`, `ordem.construtiva_gulosa`, `ordem.construtiva_gulosa_aleatoria` |
| movimento `(solucao, itens, C, k) → solucao` | `binaria.gerar_vizinho` | `ordem.gerar_vizinho` |
| perturbação `(solucao, itens, C, k) → solucao` | `binaria.perturbar` | `ordem.perturbar` |
| busca local `(solucao, itens, C) → (solucao, avaliações)` | `binaria.busca_local_melhor_melhora`, `binaria.busca_local_primeira_melhora` | `ordem.busca_local_primeira_melhora` |
| remoção (ALNS) `(solucao, itens, C, k) → solucao` | `binaria.remocao_aleatoria`, `binaria.remocao_pior`, `binaria.remocao_shaw` | `ordem.remocao_aleatoria`, `ordem.remocao_pior`, `ordem.remocao_shaw` |
| inserção (ALNS) `(solucao, itens, C) → solucao` | `binaria.insercao_aleatoria`, `binaria.insercao_gulosa` | `ordem.insercao_aleatoria`, `ordem.insercao_gulosa`, `ordem.insercao_arrependimento` |
| vizinhança (Busca Tabu) `(solucao, itens, C) → [(rótulo, itens alterados, vizinha)]` | `binaria.vizinhanca_flip`, `binaria.vizinhanca_flip_troca` | `ordem.vizinhanca_troca` |
| reconexão (GRASP) `(solucao, itens, C, guia) → [(rótulo, vizinha)]` | `binaria.passos_para_guia` | `ordem.passos_para_guia` |
| coordenadas (PSO) `(solucao, itens, C) → [float]` | `binaria.coordenadas` (os bits) | `ordem.coordenadas` (a posição de cada item na ordem) |
| voo (PSO) `(solucao, itens, C, velocidade) → solucao` | `binaria.mover_sigmoide` | `ordem.mover_chaves` |
| cruzamento (AG) `(pai1, pai2, itens, C) → (filho1, filho2)` | `binaria.cruzamento_um_ponto`, `binaria.cruzamento_multiponto`, `binaria.cruzamento_uniforme` | `ordem.cruzamento_opx`, `ordem.cruzamento_lox`, `ordem.cruzamento_pmx` |
| mutação (AG): o encaixe do movimento | `binaria.gerar_vizinho` | `ordem.gerar_vizinho`, `ordem.mutacao_insercao`, `ordem.mutacao_inversao`, `ordem.perturbar` |
| reparo (AG) `(solucao, itens, C) → solucao` | `binaria.reparar` | — (`reparo=None`) |
| seleção (AG) `(valores, quantidade) → [índices]` | `selecao_roleta`, `selecao_ranking`, `selecao_torneio`, `selecao_classes` | (as mesmas) |
| construção da formiga (ACO) `(itens, C, feromonio, α, β, q0) → solucao` | `binaria.construtiva_formiga` (feromônio por item) | `ordem.construtiva_formiga` (feromônio por transição) |
| trilha (ACO) `(solucao, itens, C) → [componentes]` | `binaria.trilha` (os itens dentro) | `ordem.trilha` (as transições entre os itens colocados) |
| aceitação `(delta, iteração) → bool` | `aceita_melhora`, `metropolis(temperaturas(...), passo)` | (as mesmas) |

No PSO, as coordenadas transformam uma solução em números, para a equação da velocidade, e o voo aplica a velocidade: na binária, cada bit vira 1 com probabilidade sigmoide(v); na ordem, a velocidade desloca a posição de cada item. No AG, o cruzamento combina dois pais em dois filhos, e os pontos de corte opcionais reproduzem os exemplos dos slides. A mutação usa o encaixe do movimento. O reparo torna viável um filho binário que passou da capacidade e completa a mochila de forma gulosa; na ordem, todo filho já é viável. A seleção escolhe índices pela aptidão. No ACO, a formiga escolhe o próximo item entre os que cabem, com probabilidade proporcional a τ^α·η^β (η = densidade), e a trilha diz onde ela deposita feromônio: no item, na binária, ou na transição de um item para o seguinte, na ordem, como as arestas do caixeiro. A construtiva gulosa aleatória sorteia cada item numa lista restrita de candidatos (RCL): com α = 0 é a gulosa, com α = 1 é aleatória entre os que cabem. A reconexão lista os passos que trazem um atributo de uma solução guia. A avaliação penalizada (`valor − α·excesso de peso`, com α = soma dos valores) dá às soluções que não cabem um valor negativo em vez de 0. A vizinhança lista todas as vizinhas, com os itens que viram tabu. A remoção tem o mesmo encaixe da perturbação, então as duas são trocáveis. A inserção por arrependimento só existe na ordem: ela compara a melhor posição de um item com a segunda melhor, e na binária um item de fora só tem um jeito de entrar.

Abaixo, cada peça aplicada uma vez na instância correlacionada.

In [ ]:
random.seed(1)
inicial = binaria.construtiva_aleatoria(itens, capacidade)
gulosa = binaria.construtiva_gulosa(itens, capacidade)
print("construtiva aleatória:", binaria.avaliar(inicial, itens, capacidade))
print("construtiva gulosa:   ", binaria.avaliar(gulosa, itens, capacidade))

vizinho = binaria.gerar_vizinho(inicial, itens, capacidade, k=1)
perturbada = binaria.perturbar(inicial, itens, capacidade, k=4)
print("movimento (1 bit):    ", binaria.avaliar(vizinho, itens, capacidade))
print("perturbação (k=4):    ", binaria.avaliar(perturbada, itens, capacidade))

melhor, n = binaria.busca_local_melhor_melhora(inicial, itens, capacidade)
primeira, m = binaria.busca_local_primeira_melhora(inicial, itens, capacidade)
print(f"busca local melhor melhora:   {binaria.avaliar(melhor, itens, capacidade)} ({n} avaliações)")
print(f"busca local primeira melhora: {binaria.avaliar(primeira, itens, capacidade)} ({m} avaliações)")

inicial_ordem = ordem.construtiva_aleatoria(itens, capacidade)
print("ordem aleatória:", ordem.avaliar(inicial_ordem, itens, capacidade),
      "· ordem gulosa:", ordem.avaliar(ordem.construtiva_gulosa(itens, capacidade), itens, capacidade))
otima_local, k = ordem.busca_local_primeira_melhora(inicial_ordem, itens, capacidade)
print(f"busca local por ordem: {ordem.avaliar(otima_local, itens, capacidade)} ({k} avaliações)")

parcial = binaria.remocao_shaw(inicial, itens, capacidade, k=5)
reconstruida = binaria.insercao_gulosa(parcial, itens, capacidade)
print("remoção Shaw (k=5):   ", binaria.avaliar(parcial, itens, capacidade),
      "→ inserção gulosa:", binaria.avaliar(reconstruida, itens, capacidade))

cheia = [1] * len(itens)  # todos os itens: passa da capacidade
print("avaliação comum e penalizada da mochila cheia:", binaria.avaliar(cheia, itens, capacidade),
      binaria.avaliar_penalizada(cheia, itens, capacidade))
print("vizinhanças da solução inicial:", len(binaria.vizinhanca_flip(inicial, itens, capacidade)), "(1-flip),",
      len(binaria.vizinhanca_flip_troca(inicial, itens, capacidade)), "(1-flip + troca)")

for a in (0, 0.3, 1):
    random.seed(1)
    print(f"construtiva gulosa aleatória (α = {a}):", binaria.avaliar(binaria.construtiva_gulosa_aleatoria(itens, capacidade, a), itens, capacidade))
print("passos de (inicial) até a gulosa:", len(binaria.passos_para_guia(inicial, itens, capacidade, gulosa)))

random.seed(1)
print("voo binário com v = 4 em todos os itens:", sum(binaria.mover_sigmoide(inicial, itens, capacidade, [4.0] * N_ITENS)), "de", N_ITENS, "bits em 1")
print("voo por ordem, item 9 com v = +4:", ordem.mover_chaves(inicial_ordem, itens, capacidade, [4.0 if j == 9 else 0.0 for j in range(N_ITENS)])[:6], "...")

aceitar = metropolis(temperaturas(1000, 1, 0.9), 100)
print("Metropolis, piora de 50 na iteração 1 (T = 1000):", aceitar(-50, 1), aceitar.ultimo)
print("só melhora:", aceita_melhora(-50, 1))

# AG: os exemplos dos slides 25 (um ponto) e 30 (LOX), com os cortes fixos
print("um ponto (slide 25):", binaria.cruzamento_um_ponto([1, 1, 0, 1, 0, 1], [1, 0, 0, 1, 0, 0], [], 0, corte=2))
print("LOX (slide 30):", ordem.cruzamento_lox([1, 2, 3, 4, 5, 6, 7, 8, 9, 0], [9, 3, 7, 8, 2, 6, 5, 1, 4, 0], [], 0, cortes=(3, 7))[0])
random.seed(1)
filho, _ = binaria.cruzamento_uniforme(inicial, gulosa, itens, capacidade)
print("cruzamento uniforme (inicial × gulosa), (valor, peso):", binaria.calcular_valor_peso(filho, itens, capacidade),
      "→ reparado:", binaria.calcular_valor_peso(binaria.reparar(filho, itens, capacidade), itens, capacidade))
random.seed(1)
print("roleta do slide 17 (aptidões 150, 50, 200, 100), 10 giros:", selecao_roleta([150, 50, 200, 100], 10))
print("torneio binário (q = 2), 10 vencedores:      ", selecao_torneio([150, 50, 200, 100], 10))

# ACO: sem feromônio, τ = 1, e a formiga sorteia pela densidade
random.seed(1)
formiga = binaria.construtiva_formiga(itens, capacidade)
print("formiga sem feromônio:", binaria.avaliar(formiga, itens, capacidade),
      "· trilha (itens):", binaria.trilha(formiga, itens, capacidade)[:5], "...")
formiga_ordem = ordem.construtiva_formiga(itens, capacidade)
print("formiga por ordem, trilha (transições):", ordem.trilha(formiga_ordem, itens, capacidade)[:3], "...")
# o exemplo do slide 27: τ35 = (1 − 0,5)·1 + 1/9,8 + 1/9,8 + 1/12,4
tau = Feromonio(1.0)
tau.evaporar(0.5)
for comprimento in (9.8, 9.8, 12.4):
    tau[(3, 5)] += 1 / comprimento
print("feromônio da aresta (3, 5) no slide 27:", round(tau[(3, 5)], 3))

## 4. Debug: uma figura por etapa

O `Depurador` desenha o que o algoritmo enxerga em cada etapa: a solução atual (faixa no topo), o plano peso × valor com a trilha e as vizinhas, e o mini-histórico na mesma escala de valor.

- `show=True`: mostra as figuras aqui, abaixo da célula. `save=True`: grava em `plots/debug/<data-hora>/`.
- `a_cada=N`: desenha a cada N iterações (mais a primeira e a última).
- `limite`: número máximo de figuras. Um SA de 6.600 iterações com `a_cada=1` geraria 6.600 figuras.
- Na representação por ordem, passe `decodificar=ordem.decodificar`.
- No PSO, cada iteração desenha o enxame inteiro no plano peso × valor, em relação ao gbest.
- No AG, cada geração desenha a população inteira, em relação ao melhor do começo da geração; nas gerações desenhadas, cada mutação também gera a sua figura.
- No ACO, cada iteração desenha a solução de cada formiga, em relação ao melhor encontrado até o começo da iteração.
- No GRASP, as figuras mostram cada passada da busca local e cada passo do path-relinking.
- Na Busca Tabu, cada iteração desenha a vizinhança inteira, com as vizinhas tabu marcadas com "(tabu)".
- Na ALNS, cada iteração desenha a remoção e a inserção sorteadas. Uma piora rejeitada por já ter sido encontrada aparece com a nota "já encontrada".

O depurador não sorteia números, então o resultado com e sem debug é o mesmo.

In [ ]:
random.seed(3)
itens_pequenos = gerar_instancia_aleatoria(8, 20, 100)
capacidade_pequena = sum(item.peso for item in itens_pequenos) // 2
otimo_pequeno, _ = programacao_dinamica(itens_pequenos, capacidade_pequena)

debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=10, limite=4)
random.seed(7)
_ = simulated_annealing_mochila(itens_pequenos, capacidade_pequena, 50, 1, 0.5, 5,
                                debug=debug)

In [ ]:
# no ILS, as figuras incluem cada passada da busca local e a perturbação
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, limite=4)
random.seed(1)
_ = iterated_local_search_mochila(itens_pequenos, capacidade_pequena, 2, 3, debug=debug)

In [ ]:
# na ALNS, as figuras mostram a remoção e a inserção sorteadas em cada iteração
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=20, limite=7)
random.seed(1)
_ = adaptive_large_neighborhood_search_mochila(itens_pequenos, capacidade_pequena,
                                               50, 1, 0.9, debug=debug)

In [ ]:
# Busca Tabu no exemplo da aula (C = 23): partindo de (01010), com |T| = 1 e BTmax = 1,
# a busca para na iteração 5 e devolve (00111), de valor 11
itens_slide = [Item(p, v) for p, v in zip([4, 5, 7, 9, 6], [2, 2, 3, 4, 4])]
debug = Depurador(itens_slide, 23, valor_otimo=11, show=True, limite=11)
melhor_slide, _ = tabu_search_mochila(itens_slide, 23, tamanho_tabu=1, max_sem_melhora=1,
                                      construtiva=lambda *_: [0, 1, 0, 1, 0], debug=debug)
print("melhor:", melhor_slide, binaria.calcular_valor_peso(melhor_slide, itens_slide, 23))

In [ ]:
# GRASP numa instância correlacionada pequena: os ótimos locais variam, então o
# path-relinking tem caminho até a solução guia
random.seed(3)
itens_grasp = gerar_instancia_aleatoria(8, 20, 100, correlacionada=True)
capacidade_grasp = sum(item.peso for item in itens_grasp) // 2
otimo_grasp, _ = programacao_dinamica(itens_grasp, capacidade_grasp)
debug = Depurador(itens_grasp, capacidade_grasp, valor_otimo=otimo_grasp, show=True,
                  a_cada=4, limite=8)
random.seed(4)
_ = grasp_mochila(itens_grasp, capacidade_grasp, max_iteracoes=6, alfa=0.9, debug=debug)

In [ ]:
# PSO: cada figura "enxame" mostra todas as partículas em relação ao gbest
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=5, limite=4)
random.seed(5)
_ = particle_swarm_optimization_mochila(itens_pequenos, capacidade_pequena,
                                        tamanho_enxame=10, max_iteracoes=10, debug=debug)

In [ ]:
# AG: cada figura "população" mostra todos os indivíduos em relação ao melhor
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=5, limite=6)
random.seed(5)
_ = genetic_algorithm_mochila(itens_pequenos, capacidade_pequena,
                              tamanho_populacao=10, max_geracoes=10, debug=debug)

In [ ]:
# ACO: cada figura "formigas" mostra a solução de cada formiga em relação ao melhor
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=5, limite=4)
random.seed(5)
_ = ant_colony_optimization_mochila(itens_pequenos, capacidade_pequena,
                                    numero_formigas=10, max_sem_melhora=10, debug=debug)

## 5. Linha de base: SA, ILS, VNS, ALNS, Busca Tabu, GRASP, PSO, AG e ACO puros

Cada esquema roda com as peças padrão (binárias) em várias sementes, na instância correlacionada.

Compare pelas **avaliações**, não pelas iterações: uma iteração do SA ou da ALNS avalia 1 candidato, enquanto uma do ILS ou do VNS roda uma busca local inteira, com centenas de avaliações, uma da Busca Tabu avalia a vizinhança inteira (uma vizinha por item), uma geração do AG avalia cada filho (cerca de 43 com os padrões), e uma iteração do ACO avalia uma solução por formiga (20).

No gráfico da ALNS, o painel de baixo mostra o peso de cada operador, atualizado a cada segmento de 100 iterações, com a temperatura no eixo da direita. Na Busca Tabu, o painel de baixo mostra o tamanho da lista tabu, que aqui é dinâmica.

O GRASP aparece duas vezes: com path-relinking (o padrão) e puro (`reconexao=None`), para conferir o slide 34 ("GRASP + PR é melhor que GRASP puro"). No GRASP, a linha cinza do gráfico é a solução construída, e a azul, a solução depois da busca local e do PR. No PSO, a cinza é a média do enxame, e a azul, a melhor partícula de cada iteração. No AG, a cinza é a média da população, a azul, o melhor indivíduo da população, e o painel de baixo mostra quantos indivíduos distintos restam (a diversidade). No ACO, a cinza é a média das formigas, a azul, a melhor formiga de cada iteração, e o painel de baixo mostra quantas soluções distintas as formigas construíram (1 é a estagnação).

In [ ]:
SEMENTES = range(10)
PARAMETROS_SA = dict(temperatura_inicial=1000, temperatura_minima=1,
                     fator_resfriamento=0.9, iteracoes_por_temperatura=100)
PARAMETROS_ALNS = dict(temperatura_inicial=10, temperatura_minima=0.01,
                       fator_resfriamento=0.999)
PARAMETROS_BT = dict(tamanho_tabu=(5, 25), max_sem_melhora=100)
PARAMETROS_PSO = dict(tamanho_enxame=30, max_iteracoes=200)

execucoes = {
    "SA": rodar_sementes(simulated_annealing_mochila, itens, capacidade, SEMENTES,
                         **PARAMETROS_SA),
    "ILS": rodar_sementes(iterated_local_search_mochila, itens, capacidade, SEMENTES,
                          max_iteracoes=250, intensidade_perturbacao=4),
    "VNS": rodar_sementes(variable_neighborhood_search_mochila, itens, capacidade,
                          SEMENTES, max_iteracoes=250, k_max=4),
    "ALNS": rodar_sementes(adaptive_large_neighborhood_search_mochila, itens, capacidade,
                           SEMENTES, **PARAMETROS_ALNS),
    "BT": rodar_sementes(tabu_search_mochila, itens, capacidade, SEMENTES, **PARAMETROS_BT),
    "GRASP+PR": rodar_sementes(grasp_mochila, itens, capacidade, SEMENTES),
    "GRASP": rodar_sementes(grasp_mochila, itens, capacidade, SEMENTES, reconexao=None),
    "PSO": rodar_sementes(particle_swarm_optimization_mochila, itens, capacidade, SEMENTES,
                          **PARAMETROS_PSO),
    "AG": rodar_sementes(genetic_algorithm_mochila, itens, capacidade, SEMENTES),
    "ACO": rodar_sementes(ant_colony_optimization_mochila, itens, capacidade, SEMENTES),
}
linha_de_base = {rotulo: resumir(e, otimo) for rotulo, e in execucoes.items()}
print(tabela(linha_de_base))

In [ ]:
_ = plotar_comparacao({rotulo: e[0]["historico"] for rotulo, e in execucoes.items()},
                      valor_otimo=otimo, show=True)
_ = plotar_historico(execucoes["SA"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="SA, semente 0", show=True)
_ = plotar_historico(execucoes["ALNS"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="ALNS, semente 0", show=True)
_ = plotar_historico(execucoes["BT"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="Busca Tabu, semente 0", show=True)
_ = plotar_historico(execucoes["GRASP+PR"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="GRASP + PR, semente 0", show=True)
_ = plotar_historico(execucoes["PSO"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="PSO, semente 0", show=True)
_ = plotar_historico(execucoes["AG"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="AG, semente 0", show=True)
_ = plotar_historico(execucoes["ACO"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="ACO, semente 0", show=True)

## 6. Suas misturas

Para montar uma combinação:

1. escolha o esquema (o esqueleto): `simulated_annealing_mochila`, `iterated_local_search_mochila`, `variable_neighborhood_search_mochila`, `adaptive_large_neighborhood_search_mochila`, `tabu_search_mochila`, `grasp_mochila`, `particle_swarm_optimization_mochila`, `genetic_algorithm_mochila` ou `ant_colony_optimization_mochila`;
2. troque as peças passando-as como parâmetro (`avaliar`, `construtiva`, `movimento` ou `perturbacao`, `busca_local`, `aceitacao` e, na ALNS, as listas `remocoes` e `insercoes`; na Busca Tabu, a `vizinhanca`; no GRASP, a `reconexao`; no PSO, `coordenadas` e `mover`; no AG, `cruzamento`, `mutacao`, `reparo`, `selecao_pais` e `selecao_sobreviventes`; no ACO, `construtiva` e `trilha`). Todas as peças de uma execução precisam ser da mesma representação, com o mesmo prefixo (`binaria.` ou `ordem.`). Não crie uma variável chamada `ordem`: ela esconderia o módulo;
3. rode em várias sementes e compare com a linha de base na tabela e no gráfico.

O molde abaixo traz, comentados, como trocar a representação do SA para "por ordem", como usar a perturbação como remoção da ALNS, como dar à Busca Tabu a vizinhança com trocas, o PSO com inércia decrescente e com busca local, o GRASP reativo e com viés exponencial na RCL (este precisa de `from functools import partial`) e o AG por ordem (com LOX e sem reparo), sem reparo (com a avaliação penalizada), com torneio e com busca local em cada filho (perto de 1 s por semente), e o ACO com β = 2, EAS, ranking, ACS, por ordem (transições) e com busca local. Na representação por ordem com busca local, prefira instâncias de até ~50 itens: cada vizinho é decodificado do zero, e com 100 itens uma busca local leva perto de 1 s. O mesmo vale para a inserção gulosa e por arrependimento da ALNS por ordem.

In [ ]:
minhas_misturas = {
    # "SA por ordem": (simulated_annealing_mochila, dict(
    #     **PARAMETROS_SA,
    #     avaliar=ordem.avaliar, construtiva=ordem.construtiva_aleatoria, movimento=ordem.gerar_vizinho,
    # )),
    # "ALNS com perturbação": (adaptive_large_neighborhood_search_mochila, dict(
    #     **PARAMETROS_ALNS,
    #     remocoes=(binaria.remocao_shaw, binaria.perturbar),
    # )),
    # "BT com trocas": (tabu_search_mochila, dict(
    #     **PARAMETROS_BT,
    #     vizinhanca=binaria.vizinhanca_flip_troca,
    # )),
    # "PSO inércia (1, 0.4)": (particle_swarm_optimization_mochila, dict(
    #     **PARAMETROS_PSO, inercia=(1.0, 0.4),
    # )),
    # "PSO + busca local": (particle_swarm_optimization_mochila, dict(
    #     **PARAMETROS_PSO, busca_local=binaria.busca_local_primeira_melhora,
    # )),
    # "GRASP reativo": (grasp_mochila, dict(alfa=(0.1, 0.3, 0.5, 0.7, 0.9))),
    # "GRASP viés exponencial": (grasp_mochila, dict(
    #     construtiva=partial(binaria.construtiva_gulosa_aleatoria, vies="exponencial"),
    # )),
    # "AG por ordem": (genetic_algorithm_mochila, dict(
    #     avaliar=ordem.avaliar, construtiva=ordem.construtiva_aleatoria,
    #     cruzamento=ordem.cruzamento_lox, mutacao=ordem.mutacao_inversao, reparo=None,
    # )),
    # "AG sem reparo": (genetic_algorithm_mochila, dict(
    #     reparo=None, avaliar=binaria.avaliar_penalizada,
    # )),
    # "AG torneio": (genetic_algorithm_mochila, dict(
    #     selecao_pais=selecao_torneio, selecao_sobreviventes=selecao_torneio,
    # )),
    # "AG + busca local": (genetic_algorithm_mochila, dict(
    #     busca_local=binaria.busca_local_primeira_melhora,
    # )),
    # "ACO β = 2": (ant_colony_optimization_mochila, dict(beta=2)),
    # "EAS": (ant_colony_optimization_mochila, dict(beta=2, deposito="elitista")),
    # "ASrank": (ant_colony_optimization_mochila, dict(beta=2, deposito="ranking")),
    # "ACS": (ant_colony_optimization_mochila, dict(
    #     beta=2, deposito="melhor", q0=0.9, atualizacao_local=0.1, evaporacao=0.1,
    # )),
    # "ACO por ordem": (ant_colony_optimization_mochila, dict(
    #     avaliar=ordem.avaliar, construtiva=ordem.construtiva_formiga, trilha=ordem.trilha,
    # )),
    # "ACO + busca local": (ant_colony_optimization_mochila, dict(
    #     beta=2, busca_local=binaria.busca_local_primeira_melhora,
    # )),
}

resumos = dict(linha_de_base)
historicos = {rotulo: e[0]["historico"] for rotulo, e in execucoes.items()}
for rotulo, (esquema, parametros) in minhas_misturas.items():
    execucoes_mistura = rodar_sementes(esquema, itens, capacidade, SEMENTES, **parametros)
    resumos[rotulo] = resumir(execucoes_mistura, otimo)
    historicos[rotulo] = execucoes_mistura[0]["historico"]

print(tabela(resumos))
_ = plotar_comparacao(historicos, valor_otimo=otimo, show=True)